In [ ]:
import numpy as np
from scipy.optimize import root_scalar
from datetime import datetime, timezone
import spiceypy as sp
# Make sure these are downloaded onn your device, I downloaded these from HW3
sp.furnsh("de430.bsp")
sp.furnsh("naif0012.tls")

In [ ]:
# Constants
mu = 0.0002959122082855911   # Sun gravitational parameter [au^3/day^2]
au_km = 1.495978707e8        # km in 1 au
day_sec = 86400.0            # seconds in 1 day

# Earth position [km] and velocity [km/s] from Spiceypy
rE_spice_km = np.array([-1.25280005e+08, -7.64193198e+07, -3.31212804e+07])
vE_spice_kms = np.array([15.9990989, -22.8704682, -9.91299077])

def date_to_mjd(year, month, day):
    # Julian Date reference: 1858-11-17 = MJD 0
    dt = datetime(year, month, day, tzinfo=timezone.utc)

    # Convert to Julian Date
    unix_epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    seconds = (dt - unix_epoch).total_seconds()

    jd = 2440587.5 + seconds / 86400.0
    # Convert to Modified Julian Date
    mjd = jd - 2400000.5
    return mjd

# Impact date of Earth and PDC25 in MJD
t_impact = date_to_mjd(2041, 4, 24)
print(t_impact)

# Convert spice date to au/day
def spice_to_au_day(r_km, v_kms):
    r_au = r_km / au_km
    v_au_day = v_kms * day_sec / au_km
    return r_au, v_au_day

rE_ref, vE_ref = spice_to_au_day(rE_spice_km, vE_spice_kms)

# PDC25 Orbital Elements
q_ast = 1.00538004981                   # au
e_ast = 0.390657994905
i_ast = np.deg2rad(10.6888293122)
RAAN_ast = np.deg2rad(214.423769139)
w_ast = np.deg2rad(359.963802784)
tp_ast = 60440.5982662                  # MJD

# Solve for semi-major axis
a_ast = q_ast / (1 - e_ast)

# Kepler Solver
def solve_kepler(M, e, tol=1e-8):
    # Iterate for Eccentric Anomaly
    E = M
    for _ in range(100):
        f = E - e*np.sin(E) - M
        fp = 1 - e*np.cos(E)
        E_new = E - f/fp
        if abs(E_new - E) < tol:
            return E_new
        E = E_new
    return E

# Convert orbital elements to state vector
def oe_to_rv(a, e, i, RAAN, w, M):

    # Solve Kepler’s equation
    E = solve_kepler(M, e)

    # True anomaly
    nu = 2*np.arctan2(np.sqrt(1 + e)*np.sin(E/2), np.sqrt(1 - e)*np.cos(E/2))

    # Radius magnitude
    r_mag = a*(1 - e*np.cos(E))

    # Position in perifocal frame
    r_pf = np.array([r_mag*np.cos(nu), r_mag*np.sin(nu), 0])

    # Velocity in perifocal frame
    v_pf = np.array([-np.sin(E), np.sqrt(1 - e**2)*np.cos(E), 0])*np.sqrt(mu*a)/r_mag
    
    # Rotation Matrix Elements
    R3_RAAN = np.array([[np.cos(RAAN), -np.sin(RAAN), 0],
                        [np.sin(RAAN), np.cos(RAAN), 0],
                        [0, 0, 1]])
    
    R1_i = np.array([[1, 0, 0],
                     [0, np.cos(i), -np.sin(i)],
                     [0, np.sin(i), np.cos(i)]])
    
    R3_w = np.array([[np.cos(w), -np.sin(w), 0],
                     [np.sin(w), np.cos(w), 0],
                     [0, 0, 1]])
    
    # Rotation Matrix
    R = R3_RAAN @ R1_i @ R3_w

    r = R @ r_pf
    v = R @ v_pf

    return r, v

# Two body propagation
def propagate_two_body(r0, v0, dt):
    r = r0.copy()
    v = v0.copy()

    steps = 200
    dt_step = dt / steps

    for _ in range(steps):
        r_mag = np.linalg.norm(r)
        a = (-mu*r)/(r_mag**3)
        v += a*dt_step
        r += v*dt_step

    return r, v

# Propagate Earth's state from Spice impact
def earth_state(t):
    dt = t - t_impact
    return propagate_two_body(rE_ref, vE_ref, dt)

# Propagate PDC25's state
def asteroid_state(t):
    n = np.sqrt(mu/(a_ast**3))
    M = n*(t - tp_ast)
    return oe_to_rv(a_ast, e_ast, i_ast, RAAN_ast, w_ast, M)

# Stumpff Functions for Lambert Solver (Chapter 6)
def C(z):
    if z > 0:
        return (1 - np.cos(np.sqrt(z)))/z
    elif z < 0:
        return (np.cosh(np.sqrt(-z)) - 1)/(-z)
    else:
        return 0.5

def S(z):
    if z > 0:
        return (np.sqrt(z) - np.sin(np.sqrt(z)))/((z)**1.5)
    elif z < 0:
        return (np.sinh(np.sqrt(-z)) - np.sqrt(-z))/((-z)**1.5)
    else:
        return 1/6

# Lambert Solver (Chapter 6)
def lambert(r1, r2, dt):
    r1_mag = np.linalg.norm(r1)
    r2_mag = np.linalg.norm(r2)

    # Solve for dtheta in the correct quadrant
    def transfer_angle(r1, r2, prograde=True):
        r1_mag = np.linalg.norm(r1)
        r2_mag = np.linalg.norm(r2)

        cos_dtheta = np.dot(r1, r2) / (r1_mag * r2_mag)
        # Ensure the cos values are within -1 and 1
        cos_dtheta = np.clip(cos_dtheta, -1.0, 1.0)

        theta = np.arccos(cos_dtheta)

        cross_z = np.cross(r1, r2)[2]

        if prograde:
            if cross_z >= 0:
                return theta
            else:
                return 2*np.pi - theta
        else:
            if cross_z < 0:
                return theta
            else:
                return 2*np.pi - theta

    # Assuming prograde trajectory
    dtheta = transfer_angle(r1, r2, prograde=True)

    cos_dtheta = np.dot(r1, r2) / (r1_mag * r2_mag)
    # Ensure the cos values are within -1 and 1
    cos_dtheta = np.clip(cos_dtheta, -1.0, 1.0)

    # Denominator of A --> don't solve is dtheta is too small
    denom = 1 - cos_dtheta
    if abs(denom) < 1e-12:
        raise ValueError("Degenerate transfer angle")

    A = np.sin(dtheta) * np.sqrt((r1_mag * r2_mag) / denom)

    # Solve for z using root finder
    def F(z):
        Cz = C(z)
        Sz = S(z)

        # Avoid invalid geometry
        if Cz <= 0:
            return 1e9

        y = r1_mag + r2_mag + (A*(z*Sz - 1))/np.sqrt(Cz)

        if y <= 0 or np.isnan(y):
            return 1e9

        dt_z = (((y/Cz)**1.5)*Sz + A*np.sqrt(y))/np.sqrt(mu)
        return dt_z - dt

    try:
        sol = root_scalar(F, bracket=[-4, 4], method='bisect')
        z = sol.root
    except:
        raise ValueError("Lambert solver failed to converge")

    Cz = C(z)
    Sz = S(z)

    y = r1_mag + r2_mag + (A*(z*Sz - 1))/np.sqrt(Cz)

    if y <= 0 or np.isnan(y):
        raise ValueError("Invalid trajectory (y <= 0)")

    f = 1 - y / r1_mag
    g = A*np.sqrt(y/mu)
    gdot = 1 - y/r2_mag

    v1 = (r2 - f*r1)/g
    v2 = (gdot*r2 - r1)/g

    return v1, v2